In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4')
small_llm = ChatOpenAI(model='gpt-5.4-mini')

In [9]:
%pip install -qU langchain-community langchain

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -qU  yfinance


Note: you may need to restart the kernel to use updated packages.


In [10]:
from typing import Literal
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState
from langgraph.types import Command


market_research_agent = create_agent(
    llm, 
    tools=[YahooFinanceNewsTool()], 
    system_prompt='You are a market researcher. provide fact only not opinions'
    )

def market_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = market_research_agent.invoke(state)
    # print(f'market_research_node: {result}')
    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='market_research')]},
        goto='supervisor'
    )

In [11]:
import yfinance as yf
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> dict:
    """Given a stock ticker, return the price date for the past month"""
    stock_info = yf.download(ticker, period='1mo').to_dict()
    return stock_info

stock_research_tools = [get_stock_price]
stock_research_agent = create_agent(
    llm, tools=stock_research_tools, system_prompt='You are a stock researcher. provide fact only not opinions'
)

def stock_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = stock_research_agent.invoke(state)
    # print(f'stock_research_node: {result}')
    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='stock_research')]}
    )

In [ ]:
@tool
def company_research_tool(ticker: str) -> dict:
    """Given a ticker, return  the financial information and SEC filings"""
    company_info = yf.Ticker(ticker)
    financial_info = company_info.get_financials()
    sec_filings = company_info.get_sec_filings()
    return {
        'financial_info': financial_info,
        'sec_filings': sec_filings
    }

company_research_tools = [company_research_tool]
company_research_agent = create_agent(
    llm, tools=company_research_tools, system_prompt='You are a company researcher. Provide facts only not opinions'
)

def company_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = company_research_agent.invoke(state)
    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='company_research')]},
        goto='supervisor'
    )
    

In [13]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import MessagesState, END
from langgraph.types import Command

members = ["market_research", "stock_research", "company_research"]

options = members + ["FINISH"]

system_prompt = {
    "You are a supervisor tasked with managing a conversation between the"
    f"following workers: {members}. Given the following user request,"
    "respond with the worker to act next. Each worker will perform a"
    "task and respond with their results and status. When finished,"
    "respond with FINISH."
}

class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH."""

    next: Literal[*options]

def supervisor_node(state: MessagesState) -> Command[Literal[*members, "__end__"]]:
    messages = [
        {"role": "system", "content": system_prompt},
    ] + state['messages']
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    if goto == "FINISH":
        goto = END

    return Command(goto=goto)

In [ ]:
# get_stock_price('AAPL')

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(MessagesState)

graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_node("market_research", market_research_node)
graph_builder.add_node("stock_research", stock_research_node)
graph_builder.add_node("company_research", company_research_node)
